# Growing Sensor Analysis

This notebook isolates the growing sensor analysis from the compost analysis and adds:

- sensor-level uptime and zero-rate profiling,
- rolling active-sensor tracking,
- sensor timelines,
- correlation analysis using active readings only, and
- a weekly moisture heatmap to spot persistent gaps,
- a sensor uptime matrix for outage and recovery phases, and
- a sensor health quadrant for fast prioritization.


In [ ]:
# Load the libraries used in the sensor notebook.
# Apply the same dark presentation theme so the sensor and compost notebooks feel like one analysis package.

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from cycler import cycler

BG = "#07111f"
PANEL = "#0d1b2a"
GRID = "#24415f"
FG = "#f3f7ff"
ACCENT = ["#00e5ff", "#ff4ecd", "#9bff66", "#ffcf40", "#7aa2ff", "#ff7b72"]

sns.set_theme(style="dark", context="talk")
sns.set_palette(ACCENT)
plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "figure.facecolor": BG,
        "axes.facecolor": PANEL,
        "axes.edgecolor": FG,
        "axes.labelcolor": FG,
        "axes.titlecolor": FG,
        "axes.prop_cycle": cycler(color=ACCENT),
        "xtick.color": FG,
        "ytick.color": FG,
        "text.color": FG,
        "grid.color": GRID,
        "grid.alpha": 0.35,
        "legend.facecolor": PANEL,
        "legend.edgecolor": GRID,
        "legend.framealpha": 0.85,
        "savefig.facecolor": BG,
        "savefig.edgecolor": BG,
    }
)


In [ ]:
# Read the combined measurement file and isolate the 12 growing-sensor moisture signals.
# Rename the columns into short sensor labels to make the later plots easier to read.

file_path = "../data/combined_compost_measurements.csv"

df = pd.read_csv(file_path, parse_dates=["Day"])

sensor_df = (
    df.rename(
        columns={
            "Day": "day",
            "Growing-Sensor 01-Moisture": "Sensor 01",
            "Growing-Sensor 02-Moisture": "Sensor 02",
            "Growing-Sensor 03-Moisture": "Sensor 03",
            "Growing-Sensor 04-Moisture": "Sensor 04",
            "Growing-Sensor 05-Moisture": "Sensor 05",
            "Growing-Sensor 06-Moisture": "Sensor 06",
            "Growing-Sensor 07-Moisture": "Sensor 07",
            "Growing-Sensor 08-Moisture": "Sensor 08",
            "Growing-Sensor 09-Moisture": "Sensor 09",
            "Growing-Sensor 10-Moisture": "Sensor 10",
            "Growing-Sensor 11-Moisture": "Sensor 11",
            "Growing-Sensor 12-Moisture": "Sensor 12",
        }
    )[
        [
            "day",
            "Sensor 01",
            "Sensor 02",
            "Sensor 03",
            "Sensor 04",
            "Sensor 05",
            "Sensor 06",
            "Sensor 07",
            "Sensor 08",
            "Sensor 09",
            "Sensor 10",
            "Sensor 11",
            "Sensor 12",
        ]
    ]
    .sort_values("day")
    .reset_index(drop=True)
)

sensor_df


## Reliability Profile

A zero reading can mean either truly dry soil or a dead/idle sensor. The summary below treats zeros as a reliability signal first, then shows the average moisture only when the sensor is actively reporting above zero.


In [ ]:
# Build a reliability summary for each sensor.
# This calculates how often each sensor is zero, how often it is active, and how long its worst outage lasts.

sensor_pd = sensor_df.copy()
sensor_cols = [column for column in sensor_pd.columns if column != "day"]

def longest_zero_streak(series: pd.Series) -> int:
    longest = 0
    current = 0

    for is_zero in series.fillna(0).eq(0):
        if is_zero:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest

sensor_summary = pd.DataFrame(
    {
        "zero_share": (sensor_pd[sensor_cols] == 0).mean().round(3),
        "active_share": (sensor_pd[sensor_cols] > 0).mean().round(3),
        "mean_when_active": sensor_pd[sensor_cols].replace(0, np.nan).mean().round(2),
        "longest_zero_streak_days": [longest_zero_streak(sensor_pd[column]) for column in sensor_cols],
    }
).sort_values(["zero_share", "longest_zero_streak_days"], ascending=False)

sensor_summary


In [ ]:
# Plot the zero-rate of every sensor as a quick reliability ranking.
# Higher bars indicate sensors that are silent or failing more often than the rest.

plt.figure(figsize=(12, 6))
sns.barplot(
    data=sensor_summary.reset_index(),
    x="index",
    y="zero_share",
    hue="index",
    palette=sns.color_palette("husl", len(sensor_summary)),
    legend=False,
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share of zero readings")
plt.xlabel("Sensor")
plt.title("Zero-rate ranking across growing sensors")
plt.tight_layout()
plt.show()


## How Many Sensors Are Active At Once?

This helps distinguish between individual failures and broader downtime events that hit many sensors at the same time.


In [ ]:
# Count how many sensors report a non-zero reading each day.
# The rolling line shows whether drops in activity are isolated events or part of a wider outage period.

sensor_pd["active_sensor_count"] = (sensor_pd[sensor_cols] > 0).sum(axis=1)
sensor_pd["active_sensor_count_7d"] = sensor_pd["active_sensor_count"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sensor_pd["day"], sensor_pd["active_sensor_count"], color="#8aa4c8", alpha=0.35, label="Daily active count")
ax.plot(sensor_pd["day"], sensor_pd["active_sensor_count_7d"], color="#00e5ff", linewidth=2.5, label="7d rolling active count")
ax.set_title("Number of growing sensors reporting above zero")
ax.set_xlabel("Day")
ax.set_ylabel("Active sensors")
ax.set_ylim(0, len(sensor_cols) + 0.5)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# Reshape the sensor table into long format so each sensor can be plotted in its own panel.
# This view makes it easier to spot individual recovery patterns and persistent flatlines.

long_sensor = sensor_pd.melt(
    id_vars="day",
    value_vars=sensor_cols,
    var_name="sensor",
    value_name="moisture",
)

sensor_grid = sns.relplot(
    data=long_sensor,
    x="day",
    y="moisture",
    col="sensor",
    col_wrap=4,
    kind="line",
    height=3,
    aspect=1.2,
    facet_kws={"sharey": False},
)
sensor_grid.figure.subplots_adjust(top=0.93)
sensor_grid.figure.suptitle("Per-sensor moisture timelines")
plt.show()


Looking at the individual timelines is useful for spotting partial recoveries. Some sensors can resume normal variation after a shared outage, while others remain flat and likely need a hardware check.


In [ ]:
# Compare sensors only during periods when they are actively reporting.
# Ignoring zeros here helps reveal whether the real moisture dynamics are shared across sensors.

active_only_corr = sensor_pd[sensor_cols].replace(0, np.nan).corr()

plt.figure(figsize=(12, 10))
sns.heatmap(active_only_corr, annot=True, cmap="mako", center=0, fmt=".2f", linewidths=0.3, linecolor=GRID)
plt.title("Sensor correlation using active readings only")
plt.tight_layout()
plt.show()


In [ ]:
# Aggregate the sensor readings by week and summarize them with a median.
# The heatmap compresses long time series into a compact overview of wet, dry, and silent periods.

weekly_sensor = long_sensor.copy()
weekly_sensor["moisture"] = weekly_sensor["moisture"].replace(0, np.nan)
weekly_sensor["week"] = weekly_sensor["day"].dt.to_period("W").apply(lambda value: value.start_time)

weekly_heatmap = weekly_sensor.pivot_table(
    index="sensor",
    columns="week",
    values="moisture",
    aggfunc="median",
)

plt.figure(figsize=(16, 6))
sns.heatmap(weekly_heatmap, cmap="crest", linewidths=0.2, linecolor=GRID)
plt.title("Weekly median moisture by sensor")
plt.xlabel("Week")
plt.ylabel("Sensor")
plt.tight_layout()
plt.show()


## Sensor Uptime Matrix

This is the fastest way to spot shared outages, staggered recoveries, and permanently silent sensors. It turns the raw time series into a visual operations log for the full array.


In [ ]:
# Convert the sensor readings into a binary active-versus-inactive matrix.
# The top panel shows outages by sensor, while the bottom panel summarizes how much of the array stayed online.

activity_matrix = (sensor_pd[sensor_cols] > 0).astype(int)
activity_for_plot = activity_matrix.T

fig = plt.figure(figsize=(16, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.15)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

sns.heatmap(
    activity_for_plot,
    cmap=sns.color_palette(["#101826", "#00e5ff"], as_cmap=True),
    cbar=False,
    linewidths=0.15,
    linecolor=GRID,
    ax=ax1,
)
ax1.set_title("Sensor uptime matrix: bright bands indicate active reporting")
ax1.set_ylabel("Sensor")
ax1.set_xlabel("")
ax1.set_yticklabels(activity_for_plot.index, rotation=0)

ax2.fill_between(
    sensor_pd["day"],
    sensor_pd["active_sensor_count_7d"],
    color="#ff4ecd",
    alpha=0.35,
    label="7d rolling active count",
)
ax2.plot(sensor_pd["day"], sensor_pd["active_sensor_count"], color="#ffcf40", linewidth=1.8, alpha=0.85, label="Daily active count")
ax2.set_ylabel("Active")
ax2.set_xlabel("Day")
ax2.legend(loc="upper right")

plt.show()


## Sensor Health Quadrant

This plot turns sensor reliability into a prioritization map. Sensors further right fail more often, sensors higher up are more volatile when active, and larger markers mean longer continuous outages.


In [ ]:
# Turn the reliability summary into a prioritization chart for troubleshooting.
# Position, marker size, and color together show which sensors fail often, behave erratically, and stay offline for long stretches.

sensor_health = sensor_summary.copy()
sensor_health["sensor"] = sensor_health.index
sensor_health["active_std"] = sensor_pd[sensor_cols].replace(0, np.nan).std().reindex(sensor_health["sensor"]).values
sensor_health["mean_when_active"] = sensor_health["mean_when_active"].fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    sensor_health["zero_share"],
    sensor_health["active_std"],
    s=sensor_health["longest_zero_streak_days"] * 9 + 80,
    c=sensor_health["mean_when_active"],
    cmap="plasma",
    alpha=0.9,
    edgecolor="#d7e3ff",
    linewidth=0.8,
)

for _, row in sensor_health.iterrows():
    ax.text(
        row["zero_share"] + 0.006,
        row["active_std"] + 0.08,
        row["sensor"],
        fontsize=10,
        color=FG,
    )

ax.axvline(sensor_health["zero_share"].median(), linestyle="--", color="#00e5ff", linewidth=1.3)
ax.axhline(sensor_health["active_std"].median(), linestyle="--", color="#9bff66", linewidth=1.3)
ax.set_title("Sensor health quadrant: reliability vs. instability")
ax.set_xlabel("Share of zero readings")
ax.set_ylabel("Moisture variability when active")

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Mean moisture when active")

plt.tight_layout()
plt.show()


## Takeaways

- The sensor analysis now stands on its own and is easier to review separately from compost temperature behaviour.
- Zero-rate ranking and longest-streak summaries make hardware issues easier to prioritize.
- Active-sensor counts expose shared downtime windows that are hard to see from one sensor at a time.
- The weekly heatmap gives a compact view of where recovery happens and which sensors remain persistently silent.
- The uptime matrix makes shared failures and recovery phases immediately visible.
- The health quadrant is a fast way to identify which sensors are both unreliable and unstable enough to deserve manual inspection first.
